<div align="center">

# **Dataset restaurant's Recommendation System - Yelp**

Martina Ibarra - Elena Salomon - Florencia Nebot

MIT-gtl-uruguay-2026

---
</div>




##**Context**
This script handles the data engineering process, consolidating the Business, Users, and Reviews tables from Yelp into the final dataset for our recommendation engine.

##**Dataset business**

In [ ]:
#Kaggle access

import kagglehub

path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")

print("Path to dataset files:", path)

100%|██████████| 4.07G/4.07G [00:43<00:00, 99.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/yelp-dataset/yelp-dataset/versions/4


In [ ]:
#Import data business

import pandas as pd

business_file = f"{path}/yelp_academic_dataset_business.json"

# 1. Loading dataset by chunks
df_iter = pd.read_json(business_file, lines=True, chunksize=100000)
chunks = []

print("⏳ Processing dataset")

for chunk in df_iter:
    chunks.append(chunk)

# 2. Concatenating chunks in only one dataframe
df_business = pd.concat(chunks, ignore_index=True)



⏳ Processing dataset


In [ ]:
df_business.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150346 entries, 0 to 150345
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   business_id   150346 non-null  object 
 1   name          150346 non-null  object 
 2   address       150346 non-null  object 
 3   city          150346 non-null  object 
 4   state         150346 non-null  object 
 5   postal_code   150346 non-null  object 
 6   latitude      150346 non-null  float64
 7   longitude     150346 non-null  float64
 8   stars         150346 non-null  float64
 9   review_count  150346 non-null  int64  
 10  is_open       150346 non-null  int64  
 11  attributes    136602 non-null  object 
 12  categories    150243 non-null  object 
 13  hours         127123 non-null  object 
dtypes: float64(3), int64(2), object(9)
memory usage: 16.1+ MB


In [ ]:
import pandas as pd
import json

# 1. Convertir strings a diccionarios (si es necesario) y normalizar
# Solo aplicamos a las filas que no son nulas
df_attributes = df_business['attributes'].apply(lambda x: x if isinstance(x, dict) else {})
df_attr_flat = pd.json_normalize(df_attributes)

# 2. Unir de nuevo al dataframe principal
df_business_final = pd.concat([df_business.reset_index(drop=True), df_attr_flat], axis=1)

In [ ]:
# Seleccionamos desde la columna 14 en adelante y describimos
df_business_final.iloc[:, 14:].describe(include='all').T

,count,unique,top,freq
ByAppointmentOnly,42339,3,False,26690
BusinessAcceptsCreditCards,119765,3,True,113667
BikeParking,72638,3,True,55040
RestaurantsPriceRange2,85314,5,2,48581
CoatCheck,5584,3,False,5141
RestaurantsTakeOut,59857,3,True,52943
RestaurantsDelivery,56282,3,True,32146
Caters,40127,3,True,22337
WiFi,56914,7,u'free',27029
BusinessParking,91085,165,"{'garage': False, 'street': False, 'validated'...",34137


Primero eliminamos las columnas de variables que no consideramos relevantes o que tienen poca información. Por ejemplo: goodfordancing, tenemos info para poco más del 10% de los business y solo 1000 cumplen con esta condicion.

In [ ]:
# 1. Lista de columnas a eliminar según el análisis de describe()
cols_to_remove = [
    'HairSpecializesIn',
    'AcceptsInsurance',
    'AgesAllowed',
    'Open24Hours',
    'DietaryRestrictions',
    'BYOBCorkage',
    'BYOB',
    'Corkage',
    'BusinessAcceptsBitcoin',
    'GoodForDancing',
    'Caters',
    'CoatCheck',
    'RestaurantsCounterService' # También tiene muy pocos datos (192)
]

# 2. Aplicamos la eliminación
# axis=1 indica que son columnas
df_business_final = df_business_final.drop(columns=cols_to_remove, errors='ignore')

print(f"Columnas eliminadas. El dataset ahora tiene {df_business_final.shape[1]} columnas.")

Columnas eliminadas. El dataset ahora tiene 40 columnas.


Ahora convertimos a binarias todas las variables que consideremos

In [ ]:
# Lista de columnas que queremos pasar a 1 y 0
cols_to_binary = [
    'ByAppointmentOnly', 'BikeParking', 'RestaurantsTakeOut',
    'RestaurantsDelivery', 'Caters', 'WheelchairAccessible',
    'HappyHour', 'OutdoorSeating', 'HasTV', 'RestaurantsReservations',
    'DogsAllowed', 'GoodForKids', 'RestaurantsGoodForGroups', 'GoodForDancing',
    'RestaurantsTableService', 'DriveThru', 'BusinessAcceptsCreditCards'
]

# Definimos el mapeo (incluyendo los strings 'True'/'False' que vienen de Yelp)
binary_map = {
    True: 1, 'True': 1, 'u\'true\'': 1, 'u"true"': 1,
    False: 0, 'False': 0, 'u\'false\'': 0, 'u"false"': 0,
    'None': 0, 0: 0
}

for col in cols_to_binary:
    if col in df_business_final.columns:
        # Reemplazamos y aseguramos que el resto sea 0 (como los NaNs)
        df_business_final[col] = df_business_final[col].map(binary_map).fillna(0).astype(int)

print("Conversión binaria lista.")

Conversión binaria lista.


In [ ]:
# 1. Obtenemos el describe de las columnas desde la 14 en adelante
desc = df_business_final.iloc[:, 14:].describe(include='all').T

# 2. Filtramos aquellas donde el número de valores únicos es mayor a 2
# Esto nos dará las variables categóricas complejas (WiFi, Alcohol, PriceRange, etc.)
non_binary_cols = desc[desc['unique'] > 2].index.tolist()

print("Variables que NO son binarias y necesitan tratamiento especial (One-Hot Encoding):")
print(non_binary_cols)

# 3. Mostramos el detalle de estas columnas para ver qué valores tienen
df_business_final[non_binary_cols].describe(include='all').T

Variables que NO son binarias y necesitan tratamiento especial (One-Hot Encoding):
['RestaurantsPriceRange2', 'WiFi', 'BusinessParking', 'Alcohol', 'RestaurantsAttire', 'Ambience', 'NoiseLevel', 'GoodForMeal', 'Smoking', 'Music', 'BestNights']


,count,unique,top,freq
RestaurantsPriceRange2,85314,5,2,48581
WiFi,56914,7,u'free',27029
BusinessParking,91085,165,"{'garage': False, 'street': False, 'validated'...",34137
Alcohol,43189,7,u'none',15977
RestaurantsAttire,39255,7,u'casual',23190
Ambience,44279,2414,"{'touristy': False, 'hipster': False, 'romanti...",6717
NoiseLevel,37993,9,u'average',21581
GoodForMeal,29087,518,"{'dessert': False, 'latenight': False, 'lunch'...",9082
Smoking,4567,6,u'no',2375
Music,7521,167,"{'dj': False, 'background_music': False, 'no_m...",2836


In [ ]:
df_business_final.WiFi.value_counts()

,count
WiFi,
u'free',27029
u'no',15221
'free',7385
'no',6610
u'paid',486
'paid',133
None,50


In [ ]:
df_business_final.Smoking.value_counts()

,count
Smoking,
u'no',2375
u'outdoor',1800
u'yes',331
'no',30
'outdoor',17
None,14


In [ ]:
df_business_final.Alcohol.value_counts()

,count
Alcohol,
u'none',15977
u'full_bar',12968
'none',4933
u'beer_and_wine',4880
'full_bar',3024
'beer_and_wine',1369
None,38


Estas tres variables tambien las podemos modificar para convertirlas en binarias

In [ ]:
import pandas as pd
import numpy as np

# 1. Definimos una función de limpieza para valores inconsistentes
def to_binary(val):
    val = str(val).lower().replace("u'", "").replace("'", "").strip()
    # Si el valor indica presencia del servicio
    if val in ['true', '1', 'free', 'paid', 'full_bar', 'beer_and_wine', 'yes', 'outdoor']:
        return 1
    # Si el valor indica ausencia o es nulo
    else:
        return 0

# 2. Aplicamos la conversión a las columnas específicas
cols_to_binary = ['WiFi', 'Alcohol', 'Smoking']

for col in cols_to_binary:
    if col in df_business_final.columns:
        df_business_final[col] = df_business_final[col].apply(to_binary)

# 3. Verificamos el resultado
print(df_business_final[cols_to_binary].head())

   WiFi  Alcohol  Smoking
0     0        0        0
1     0        0        0
2     0        0        0
3     1        0        0
4     0        0        0


In [ ]:
import ast

def es_weekend(val):
    if pd.isna(val) or val == 'None': return 0
    try:
        # Convertimos el string a diccionario real
        d = ast.literal_eval(val) if isinstance(val, str) else val
        # Si 'friday' o 'saturday' son True, devolvemos 1
        if d.get('friday') or d.get('saturday'):
            return 1
        else:
            return 0
    except:
        return 0

# Creamos tu nueva variable
df_business_final['BestNight_Weekend'] = df_business_final['BestNights'].apply(es_weekend)

# Ahora podemos eliminar la columna original que era un lío
df_business_final.drop(columns=['BestNights'], inplace=True)

In [ ]:
print("\n--- 📊 Descriptive statistics for numerical features ---")
summary_num = df_business_final.describe().T
summary_num


--- 📊 Descriptive statistics for numerical features ---


,count,mean,std,min,25%,50%,75%,max
latitude,150346.0,36.671150,5.872759,27.555127,32.187293,38.777413,39.954036,53.679197
longitude,150346.0,-89.357339,14.918502,-120.095137,-90.357810,-86.121179,-75.421542,-73.200457
stars,150346.0,3.596724,0.974421,1.000000,3.000000,3.500000,4.500000,5.000000
review_count,150346.0,44.866561,121.120136,5.000000,8.000000,15.000000,37.000000,7568.000000
is_open,150346.0,0.796150,0.402860,0.000000,1.000000,1.000000,1.000000,1.000000
ByAppointmentOnly,150346.0,0.103821,0.305029,0.000000,0.000000,0.000000,0.000000,1.000000
BusinessAcceptsCreditCards,150346.0,0.756036,0.429473,0.000000,1.000000,1.000000,1.000000,1.000000
BikeParking,150346.0,0.366089,0.481736,0.000000,0.000000,0.000000,1.000000,1.000000
RestaurantsTakeOut,150346.0,0.352141,0.477639,0.000000,0.000000,0.000000,1.000000,1.000000
RestaurantsDelivery,150346.0,0.213813,0.409998,0.000000,0.000000,0.000000,0.000000,1.000000


Our dataset includes 150,346 shops, 79% of which are currently open. On average, these businesses have a 3.59-star rating and 44 reviews on the platform.

In [ ]:
df_business_final = df_business_final[
    (df_business_final['is_open'] == 1) &
  #  (df_business['city'].isin(top_5_cities)) &
    (df_business_final['categories'].str.contains('Restaurants', case=False, na=False))
].copy()

In [ ]:
print("\n--- 📊 Descriptive statistics for numerical features ---")
summary_num = df_business_final.describe().T
summary_num


--- 📊 Descriptive statistics for numerical features ---


,count,mean,std,min,25%,50%,75%,max
latitude,34987.0,37.056552,6.094542,27.564457,32.206783,39.492349,39.961469,53.679197
longitude,34987.0,-87.880634,13.773586,-120.083748,-90.236681,-86.040412,-75.360327,-74.680250
stars,34987.0,3.523895,0.862661,1.000000,3.000000,3.500000,4.000000,5.000000
review_count,34987.0,104.142767,220.590683,5.000000,16.000000,40.000000,109.000000,7568.000000
is_open,34987.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
ByAppointmentOnly,34987.0,0.003087,0.055474,0.000000,0.000000,0.000000,0.000000,1.000000
BusinessAcceptsCreditCards,34987.0,0.824449,0.380443,0.000000,1.000000,1.000000,1.000000,1.000000
BikeParking,34987.0,0.525624,0.499350,0.000000,0.000000,1.000000,1.000000,1.000000
RestaurantsTakeOut,34987.0,0.873953,0.331907,0.000000,1.000000,1.000000,1.000000,1.000000
RestaurantsDelivery,34987.0,0.633778,0.481778,0.000000,0.000000,1.000000,1.000000,1.000000


In [ ]:
import plotly.express as px
import pandas as pd

# 1. Agrupamos directamente por la columna 'city'
# Usamos el promedio de latitud y longitud para ubicar el punto en el mapa por ciudad
df_city_counts = df_business_final.groupby('city').agg(
    unique_count=('business_id', 'nunique'),
    latitude=('latitude', 'mean'),
    longitude=('longitude', 'mean')
).reset_index()

# 2. Tomamos el Top 15 de ciudades con más restaurantes
df_top_15 = df_city_counts.sort_values(by='unique_count', ascending=False).head(15)

# 3. Creamos el mapa con Plotly
fig = px.scatter_geo(
    df_top_15,
    lat='latitude',
    lon='longitude',
    hover_name='city',
    hover_data={'unique_count': True, 'latitude': False, 'longitude': False},
    color='city',
    size='unique_count',
    size_max=20, # Un poco más grande para mejor visibilidad
    scope='usa',
    title='Top 15 Cities by Restaurant Volume',
    template='plotly_white',
    width=800,
    height=600,
)

# 4. Ajustes estéticos
fig.update_layout(
    title_x=0.5,
    geo=dict(
        showland=True,
        landcolor="rgb(243, 243, 243)",
        subunitcolor="rgb(204, 204, 204)",
        showlakes=True,
        lakecolor="white"
    )
)

fig.show()

Observations: we are only keeping Restaurants (to focus only in one type of business) from top 5 cities with more restaurants and that are currently open. It doesn't make any sense to recommend or consider for the recommendation system stores that are not available to the public anymore.

In [ ]:
df_business_final = df_business_final[
    (df_business_final['is_open'] == 1) &
   # (df_business['city'].isin(top_5_cities)) &
    (df_business_final['categories'].str.contains('Restaurants', case=False, na=False))
].copy()

In [ ]:
print(f"Open restaurants: {len(df_business_final):,}")
print(f"Average rating restaurants: {df_business_final['stars'].mean():,}")
print(f"Average count reviews restaurants: {df_business_final['review_count'].mean():,}")

Open restaurants: 34,987
Average rating restaurants: 3.523894589418927
Average count reviews restaurants: 104.14276731357361


In [ ]:
df_business_final.categories.value_counts()

,count
categories,
"Restaurants, Pizza",668
"Pizza, Restaurants",575
"Restaurants, Chinese",537
"Chinese, Restaurants",513
"Restaurants, Mexican",447
...,...
"American (Traditional), Restaurants, Seafood, Steakhouses",1
"Cheesesteaks, Restaurants, Soul Food, Halal, American (Traditional)",1
"Breakfast & Brunch, Seafood, Buffets, Chinese, Restaurants",1


There are 22,304 unique restaurant categories in the dataset. However, the problem is that each restaurant is described by a long list of specific keywords. To simplify this, we decided to create generalized subcategories for each restaurant.

In [ ]:
#Get dummies from the category list and keep the most frequent tags

import pandas as pd

dummies = df_business_final['categories'].str.get_dummies(sep=', ')

tag_count = dummies.sum().sort_values(ascending=False)

top_tags = tag_count[tag_count > 50].index.tolist()
df_dummies_filtrado = dummies[top_tags]

df_final = pd.concat([df_business_final, df_dummies_filtrado], axis=1)

In [ ]:
tag_count = dummies.sum().sort_values(ascending=False)

print("Top 20 labels:")
print(tag_count.head(20))

Top 20 labels:
Restaurants                  34987
Food                         10859
Sandwiches                    6075
Nightlife                     5779
Bars                          5567
American (Traditional)        5531
Fast Food                     5516
Pizza                         5090
Breakfast & Brunch            4415
Burgers                       4275
American (New)                3629
Mexican                       3315
Coffee & Tea                  2977
Italian                       2953
Seafood                       2479
Chicken Wings                 2335
Chinese                       2295
Event Planning & Services     2276
Salad                         2263
Delis                         1734
dtype: int64


In [ ]:
# Exclued general characteristics
ignore_general_tags = [
    'Restaurants', 'Food', 'Nightlife', 'American (Traditional)', 'American (New)', 'Event Planning & Services', 'Delis'
]


total_count = dummies.sum()
only_cuisine_type = total_count[~total_count.index.isin(ignore_general_tags)].sort_values(ascending=False)

print("Top 20 main cousine:")
print(only_cuisine_type.head(20))

Top 20 main cousine:
Sandwiches            6075
Bars                  5567
Fast Food             5516
Pizza                 5090
Breakfast & Brunch    4415
Burgers               4275
Mexican               3315
Coffee & Tea          2977
Italian               2953
Seafood               2479
Chicken Wings         2335
Chinese               2295
Salad                 2263
Cafes                 1712
Caterers              1506
Specialty Food        1358
Bakeries              1322
Desserts              1285
Sports Bars           1243
Japanese              1187
dtype: int64


In [ ]:
#Principal main cousine

main_cuisine = [
    'Pizza', 'Mexican', 'Italian', 'Chinese', 'Japanese', 'Sushi Bars',
    'Thai', 'Vietnamese', 'Steakhouses', 'Barbeque', 'Seafood',
    'Burgers', 'Chicken Wings', 'Salad', 'Bakeries', 'Desserts',
    'Cafes', 'Coffee & Tea', 'Spanish', 'Mediterranean', 'Korean','Bars','Breakfast', 'Pubs','American (Traditional)','Sandwiches','Fast Food','American (New)','Breakfast & Brunch','Event Planning & Services','Delis'
    , 'Chicken Shop','Chicken','Event Spaces','Donuts','Caribbean','Street Vendors','African','Latin American','Trucks','Caribbean','Indian','Honduran','Vegan','Southern','Puerto Rican','Ice Cream & Frozen Yogurt','Cajun/Creole', 'French','Cuban'
    , 'Gluten-Free','Cheesesteaks','Filipino','Brewpubs','Ukrainian','Diners','Asian Fusion','Greek','German','Bagels','Shopping Centers','Middle Eastern','Hot Dogs','Food Court','Canadian','Food Stands'
    ,'Beer Gardens','Canadian (New)','Active Life','Soup','Hawaiian'
]

In [ ]:
#Create subcategory

def extraer_identidad_limpia(fila):

    for identidad in main_cuisine:
        if identidad in fila.index and fila[identidad] == 1:
            return identidad
    return 'Other'

df_business_final['subcategory'] = dummies.apply(extraer_identidad_limpia, axis=1)

In [ ]:
aggregations = {
    'Asian': ['Chinese', 'Japanese', 'Sushi Bars', 'Thai', 'Vietnamese', 'Korean', 'Filipino', 'Asian Fusion'],
    'American & Fast Food': ['Burgers', 'Fast Food', 'Hot Dogs', 'Cheesesteaks', 'Diners', 'American (Traditional)', 'American (New)', 'Chicken Shop', 'Chicken', 'Southern'],
    'European': ['Italian', 'Spanish', 'French', 'German', 'Greek', 'Ukrainian'],
    'Latin & Caribbean': ['Mexican', 'Caribbean', 'Latin American', 'Honduran', 'Puerto Rican', 'Cuban'],
    'Pizza & Wings': ['Pizza', 'Chicken Wings'],
    'Cafe & Desserts': ['Cafes', 'Coffee & Tea', 'Bakeries', 'Desserts', 'Donuts', 'Ice Cream & Frozen Yogurt', 'Bagels'],
    'Pubs & Nightlife': ['Bars', 'Pubs', 'Brewpubs', 'Beer Gardens', 'Cocktail Bars'],
    'Breakfast & Deli': ['Breakfast', 'Breakfast & Brunch', 'Delis', 'Sandwiches', 'Bagels'],
    'Steaks & Grill': ['Steakhouses', 'Barbeque', 'Seafood', 'Cajun/Creole'],
    'Healthy & Specialized': ['Salad', 'Vegan', 'Gluten-Free', 'Soup'],
    'International & Others': ['Mediterranean', 'Middle Eastern', 'African', 'Indian', 'Canadian', 'Canadian (New)', 'Hawaiian', 'Food Court', 'Food Stands', 'Street Vendors', 'Trucks']
}

inverse_mapping = {item: k for k, v in aggregations.items() for item in v}

In [ ]:
# Grouping Subcategories
df_business_final['main_category'] = df_business_final['subcategory'].map(inverse_mapping).fillna('Other')

print(df_business_final['main_category'].value_counts())

main_category
Pizza & Wings             6074
American & Fast Food      5608
Asian                     4447
Cafe & Desserts           3784
Latin & Caribbean         3570
Steaks & Grill            3027
Breakfast & Deli          2264
Pubs & Nightlife          2148
European                  1314
Healthy & Specialized     1117
International & Others     913
Other                      721
Name: count, dtype: int64


In [ ]:
df_business_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 34987 entries, 3 to 150339
Data columns (total 42 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   business_id                 34987 non-null  object 
 1   name                        34987 non-null  object 
 2   address                     34987 non-null  object 
 3   city                        34987 non-null  object 
 4   state                       34987 non-null  object 
 5   postal_code                 34987 non-null  object 
 6   latitude                    34987 non-null  float64
 7   longitude                   34987 non-null  float64
 8   stars                       34987 non-null  float64
 9   review_count                34987 non-null  int64  
 10  is_open                     34987 non-null  int64  
 11  attributes                  34547 non-null  object 
 12  categories                  34987 non-null  object 
 13  hours                       31617 n

In [ ]:
# List of columns to remove
cols_to_drop = [
    'Music',
    'GoodForMeal',
    'Ambience',
    'RestaurantsAttire',
    'BusinessParking',
    'postal_code',
    'address',
    'subcategory',
    'is_open'
]

# Dropping the columns
df_business_final = df_business_final.drop(columns=cols_to_drop, errors='ignore')

# Verify the new shape and remaining columns
print(f"✅ New number of columns: {len(df_business_final.columns)}")
print(df_business_final.columns.tolist())

✅ New number of columns: 33
['business_id', 'name', 'city', 'state', 'latitude', 'longitude', 'stars', 'review_count', 'attributes', 'categories', 'hours', 'ByAppointmentOnly', 'BusinessAcceptsCreditCards', 'BikeParking', 'RestaurantsPriceRange2', 'RestaurantsTakeOut', 'RestaurantsDelivery', 'WiFi', 'WheelchairAccessible', 'HappyHour', 'OutdoorSeating', 'HasTV', 'RestaurantsReservations', 'DogsAllowed', 'Alcohol', 'GoodForKids', 'RestaurantsTableService', 'RestaurantsGoodForGroups', 'DriveThru', 'NoiseLevel', 'Smoking', 'BestNight_Weekend', 'main_category']


##**Dataset users**
We import the users dataset, specifically selecting only user_id and review_count to understand the distribution of reviews and decide which users to keep. For instance, it doesn't make sense to include users who have only left a single review.

In [ ]:
import pandas as pd

users_file = f"{path}/yelp_academic_dataset_user.json"

# 1. Loading dataset by chunks
df_iter = pd.read_json(users_file, lines=True, chunksize=100000)
chunks = []

print("⏳ Processing dataset")

for chunk in df_iter:
    chunks.append(chunk)

# 2. Concatenating chunks in only one dataframe
df_users = pd.concat(chunks, ignore_index=True)


⏳ Processing dataset


In [ ]:
df_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1987897 entries, 0 to 1987896
Data columns (total 22 columns):
 #   Column              Dtype  
---  ------              -----  
 0   user_id             object 
 1   name                object 
 2   review_count        int64  
 3   yelping_since       object 
 4   useful              int64  
 5   funny               int64  
 6   cool                int64  
 7   elite               object 
 8   friends             object 
 9   fans                int64  
 10  average_stars       float64
 11  compliment_hot      int64  
 12  compliment_more     int64  
 13  compliment_profile  int64  
 14  compliment_cute     int64  
 15  compliment_list     int64  
 16  compliment_note     int64  
 17  compliment_plain    int64  
 18  compliment_cool     int64  
 19  compliment_funny    int64  
 20  compliment_writer   int64  
 21  compliment_photos   int64  
dtypes: float64(1), int64(16), object(5)
memory usage: 333.7+ MB


In [ ]:
df_users.head(2)

,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946


In [ ]:
print("\n--- 📊 Descriptive statistics for numerical features ---")
summary_num = df_users.describe().T
print(summary_num)


--- 📊 Descriptive statistics for numerical features ---
                        count       mean         std  min  25%   50%    75%  \
review_count        1987897.0  23.394409   82.566992  0.0  2.0  5.00  17.00   
useful              1987897.0  42.296335  641.480597  0.0  0.0  3.00  13.00   
funny               1987897.0  16.970536  407.803437  0.0  0.0  0.00   2.00   
cool                1987897.0  23.792914  565.351295  0.0  0.0  0.00   3.00   
fans                1987897.0   1.465740   18.130753  0.0  0.0  0.00   0.00   
average_stars       1987897.0   3.630494    1.183337  1.0  3.0  3.88   4.56   
compliment_hot      1987897.0   1.807072   73.601841  0.0  0.0  0.00   0.00   
compliment_more     1987897.0   0.292263   12.824667  0.0  0.0  0.00   0.00   
compliment_profile  1987897.0   0.179318   15.155253  0.0  0.0  0.00   0.00   
compliment_cute     1987897.0   0.133649   11.356823  0.0  0.0  0.00   0.00   
compliment_list     1987897.0   0.063907   10.043627  0.0  0.0  0.00   0.0

Observation: 50% of Yelp users leave at least 5 reviews and 75% leave at least 17. To optimize the dataset, we will keep only those users with 5 or more reviews to ensure we have enough data points for each user.

In [ ]:
df_users_final = df_users[df_users['review_count'] >= 100]

In [ ]:
df_users_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 91774 entries, 0 to 1987854
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   user_id             91774 non-null  object 
 1   name                91774 non-null  object 
 2   review_count        91774 non-null  int64  
 3   yelping_since       91774 non-null  object 
 4   useful              91774 non-null  int64  
 5   funny               91774 non-null  int64  
 6   cool                91774 non-null  int64  
 7   elite               91774 non-null  object 
 8   friends             91774 non-null  object 
 9   fans                91774 non-null  int64  
 10  average_stars       91774 non-null  float64
 11  compliment_hot      91774 non-null  int64  
 12  compliment_more     91774 non-null  int64  
 13  compliment_profile  91774 non-null  int64  
 14  compliment_cute     91774 non-null  int64  
 15  compliment_list     91774 non-null  int64  
 16  complim

The number of users decreased from 1,987,897 to 1,087,094

In [ ]:
# Identify colums with 'compliment'
cols_to_drop = [col for col in df_users_final.columns if col.startswith('compliment')]

# Delete 'compliment' columns
df_users_final = df_users_final.drop(columns=cols_to_drop)

In [ ]:
# Create 'elite_count'
df_users_final['elite_count'] = df_users_final['elite'].apply(
    lambda x: len(x.split(',')) if isinstance(x, str) and x.strip() != "" else 0
)


In [ ]:
# Create a index of user influence
df_users_final['raw_influence_score'] = (df_users_final['elite_count'] * 10) + \
                                 (df_users_final['fans'] * 5) + \
                                 (df_users_final['funny'] + df_users_final['cool'])

max_score = df_users_final['raw_influence_score'].max()
df_users_final['influence_index'] = (df_users_final['raw_influence_score'] / max_score) * 100

print(df_users_final[['name', 'elite_count', 'fans', 'influence_index']].sort_values(by='influence_index', ascending=False).head(10))

            name  elite_count  fans  influence_index
17169        Fox            9  3493       100.000000
207385    Harald           10   880        94.559510
200787   Richard           12  3243        80.010621
4723       Bruce           13   867        62.068629
207620  Marianne            7  1021        55.932484
223840     Ariel            6   808        55.336530
795339    Victor           15  1462        54.602297
800350    Rohlin            8   605        53.623570
842189    Arshad            9   597        53.345508
2216     Michael           13  1563        53.321710


In [ ]:
import pandas as pd
import numpy as np

# Create the user type based on index influence
bins = [-1, 5, 30, 80, 100]
user_type = ['Spectator', 'Active member', 'Local influencer', 'Legendary Tastemaker']

df_users_final['user_type'] = pd.cut(df_users_final['influence_index'],
                              bins=bins,
                              labels=user_type)

print("Distribución de tipos de usuario:")
print(df_users_final['user_type'].value_counts())

Distribución de tipos de usuario:
user_type
Spectator               91295
Active member             461
Local influencer           15
Legendary Tastemaker        3
Name: count, dtype: int64


##**Dataset reviews**

In [ ]:
import pandas as pd

valid_users = set(df_users_final['user_id'])
valid_businesses = set(df_business_final['business_id'])

review_file = f"{path}/yelp_academic_dataset_review.json"
chunks_reviews = []
chunk_size = 50000

print("Filter dataset for business_id, user_id and year (2017+)...")

for chunk in pd.read_json(review_file, lines=True, chunksize=chunk_size):
    chunk['date'] = pd.to_datetime(chunk['date'])

    filtered_chunk = chunk[
        (chunk['user_id'].isin(valid_users)) &
        (chunk['business_id'].isin(valid_businesses)) &
        (chunk['date'].dt.year >= 2017)
    ]

    chunks_reviews.append(filtered_chunk)
# final dataset
df_reviews_final = pd.concat(chunks_reviews, ignore_index=True)

print(f"Done!")
print(f"Total reviews: {len(df_reviews_final):,}")

Filter dataset for business_id, user_id and year (2017+)...
Done!
Total reviews: 479,615


In [ ]:
import pandas as pd

# 1. Calculamos el score base de impacto
df_reviews_final['review_impact_score'] = (df_reviews_final['useful'] * 2) + \
                                          df_reviews_final['funny'] + \
                                          df_reviews_final['cool']

# 2. Normalización (opcional)
# Esto sirve para ver qué tan cerca está una reseña del "impacto máximo" registrado
max_impact = df_reviews_final['review_impact_score'].max()
df_reviews_final['review_impact_index'] = (df_reviews_final['review_impact_score'] / max_impact) * 100

# 3. Ver las reseñas con mayor impacto
print(df_reviews_final[['review_id', 'useful', 'funny', 'cool', 'review_impact_index']].sort_values(by='review_impact_index', ascending=False).head())

                     review_id  useful  funny  cool  review_impact_index
473853  HnnDqF0ksknHEy7B2Im9Fg     325    180   304           100.000000
320371  ehUVIpfylhrxraozNY_TDQ     227    159   207            72.310406
227300  4TAKLBW49b9DFFuGatGkLA     222    159   207            71.428571
180053  _z7eXnw1-J_47DMEA6KN9g     211    157   202            68.871252
231642  wcXZBP8Q1snWzCaSIZ7Qxg     205    150   202            67.195767


In [ ]:
# Definimos los rangos de impacto
bins_review = [-1, 2, 10, 50, float('inf')]
labels_review = ['Low Impact', 'Useful', 'Viral', 'Critical Reference']
df_reviews_final['review_quality'] = pd.cut(df_reviews_final['review_impact_score'],
                                            bins=bins_review,
                                            labels=labels_review)

# Ver distribución
print(df_reviews_final['review_quality'].value_counts())

review_quality
Low Impact            233191
Useful                176979
Viral                  60767
Critical Reference      8678
Name: count, dtype: int64


In [ ]:
df_users_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 91774 entries, 0 to 1987854
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   user_id              91774 non-null  object  
 1   name                 91774 non-null  object  
 2   review_count         91774 non-null  int64   
 3   yelping_since        91774 non-null  object  
 4   useful               91774 non-null  int64   
 5   funny                91774 non-null  int64   
 6   cool                 91774 non-null  int64   
 7   elite                91774 non-null  object  
 8   friends              91774 non-null  object  
 9   fans                 91774 non-null  int64   
 10  average_stars        91774 non-null  float64 
 11  elite_count          91774 non-null  int64   
 12  raw_influence_score  91774 non-null  int64   
 13  influence_index      91774 non-null  float64 
 14  user_type            91774 non-null  category
dtypes: category(1), float6

Alphanumeric to Integer conversion: converting user_id, business_id and review_id from long strings to integers. Maintaining numeric identifiers significantly reduces the memory footprint of the dataframe.

In [ ]:
# 1. Create a user_id Mapping Dictionary. This mapping ensures that an alphanumeric ID like "KU_O5ud..."
# is consistently assigned to the same integer (e.g., User 0) across all tables.
user_map = {original_id: i for i, original_id in enumerate(df_users_final['user_id'].unique())}

# 2. Define the transformation function for the social network data
def safe_map_friends(friends_val, mapping):
    if pd.isna(friends_val) or friends_val == "None" or not isinstance(friends_val, str):
        return []
    # mapping.get(f.strip()) busca al amigo; si no existe en nuestro dataset, devuelve None
    lista_amigos = [mapping.get(f.strip()) for f in friends_val.split(',')]
    return [int(f) for f in lista_amigos if f is not None]

# Asign the int user_id to the original user_id
df_users_final['user_id_int'] = df_users_final['user_id'].map(user_map).astype('int32')

# Translate original list of friends to their equivalent user id
df_users_final['friends_int'] = df_users_final['friends'].apply(lambda x: safe_map_friends(x, user_map))

# Delete original columns
df_users_final.drop(['user_id', 'friends'], axis=1, inplace=True)

In [ ]:
# 1. Creamos la columna numérica en el dataset de reseñas
# Usamos .map(user_map) para asegurar que la identidad sea la misma
df_reviews_final['user_id_int'] = df_reviews_final['user_id'].map(user_map)

# 2. Manejo de valores nulos (opcional pero profesional)
# Si una reseña es de un usuario que no estaba en df_users, quedará como NaN.
# Es buena práctica eliminarlos si tu modelo requiere que el usuario exista en el perfil.
df_reviews_final = df_reviews_final.dropna(subset=['user_id_int'])

# 3. Optimización de memoria: convertir a entero de 32 bits
df_reviews_final['user_id_int'] = df_reviews_final['user_id_int'].astype('int32')

# 4. Ahora puedes eliminar el string original para liberar cientos de MB
df_reviews_final.drop('user_id', axis=1, inplace=True)

In [ ]:
# 1. Crear el diccionario de mapeo para negocios
# Usamos df_business como base porque contiene el catálogo maestro de tiendas
business_map = {id_orig: i for i, id_orig in enumerate(df_business_final['business_id'].unique())}

# 2. Aplicar el mapeo al dataset de Negocios
df_business_final['business_id_int'] = df_business_final['business_id'].map(business_map).astype('int32')

# 3. APLICAR EL MISMO MAPEO AL DATASET DE RESEÑAS (df_reviews_final)
# Esto es lo que permite que la reseña "conecte" con la tienda correcta
df_reviews_final['business_id_int'] = df_reviews_final['business_id'].map(business_map)
df_reviews_final.drop('business_id', axis=1, inplace=True)

# 4. Limpieza: Eliminar reseñas de negocios que no están en nuestro catálogo maestro
df_reviews_final = df_reviews_final.dropna(subset=['business_id_int'])
df_reviews_final['business_id_int'] = df_reviews_final['business_id_int'].astype('int32')

In [ ]:
# 1. Crear el diccionario de mapeo para las reseñas
# Nota: Las reseñas suelen ser únicas, por lo que el mapeo es 1 a 1 con las filas
review_map = {id_orig: i for i, id_orig in enumerate(df_reviews_final['review_id'].unique())}

# 2. Aplicar el mapeo al dataset de reseñas
df_reviews_final['review_id_int'] = df_reviews_final['review_id'].map(review_map).astype('int32')

# Delete original ID
df_reviews_final.drop('review_id', axis=1, inplace=True)

# Check results
print(df_reviews_final[['review_id_int', 'user_id_int', 'business_id_int', 'stars']].head())

   review_id_int  user_id_int  business_id_int  stars
0              0        22392             3355      5
1              1        13301              939      5
2              2        21483             1196      4
3              3        14528             1730      5
4              4        20074             1693      3


In [ ]:
df_reviews_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 479615 entries, 0 to 479614
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   stars                479615 non-null  int64         
 1   useful               479615 non-null  int64         
 2   funny                479615 non-null  int64         
 3   cool                 479615 non-null  int64         
 4   text                 479615 non-null  object        
 5   date                 479615 non-null  datetime64[ns]
 6   review_impact_score  479615 non-null  int64         
 7   review_impact_index  479615 non-null  float64       
 8   review_quality       479615 non-null  category      
 9   user_id_int          479615 non-null  int32         
 10  business_id_int      479615 non-null  int32         
 11  review_id_int        479615 non-null  int32         
dtypes: category(1), datetime64[ns](1), float64(1), int32(3), int64(5), objec

In [ ]:
df_reviews_final.head(1)

,stars,useful,funny,cool,text,date,review_impact_score,review_impact_index,review_quality,user_id_int,business_id_int,review_id_int
0,5,0,0,0,First time there and it was excellent!!! It fe...,2017-02-19 13:32:05,0,0.0,Low Impact,22392,3355,0


In [ ]:
df_business_final.head(1)

,business_id,name,city,state,latitude,longitude,stars,review_count,attributes,categories,...,Alcohol,GoodForKids,RestaurantsTableService,RestaurantsGoodForGroups,DriveThru,NoiseLevel,Smoking,BestNight_Weekend,main_category,business_id_int
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,Philadelphia,PA,39.955505,-75.155564,4.0,80,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...",...,0,0,0,0,0,NaN,0,0,Cafe & Desserts,0


In [ ]:
df_users_final.head(1)

,name,review_count,yelping_since,useful,funny,cool,elite,fans,average_stars,elite_count,raw_influence_score,influence_index,user_type,user_id_int,friends_int
0,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,267,3.91,1,8598,2.153851,Spectator,0,"[977, 25932, 89556, 1975, 277, 75631, 3585, 88..."


In [ ]:
# 1. Unir Reseñas con Restaurantes
# Usamos 'inner' para asegurar que solo queden reseñas de locales que filtramos como 'Restaurants'
df_merged = pd.merge(
    df_reviews_final,
    df_business_final,
    on='business_id_int',
    how='inner'
)

# Renombrar 'stars' de la reseña para no confundirla con el promedio del negocio
df_merged = df_merged.rename(columns={'stars': 'user_rating'})

# 2. Cargar y unir Usuarios (Opcional pero recomendado para modelos avanzados)
df_users = df_users_final
df_final = pd.merge(
    df_merged,
    df_users,
    on='user_id_int',
    how='inner'
)

print(f"Dataset final listo: {df_final.shape} interacciones.")

Dataset final listo: (479615, 59) interacciones.


In [ ]:
# Reporte detallado por columna (en Megabytes)
print(df_final.memory_usage(deep=True) / 1024**2)

# Total resumido
total_ram = df_final.memory_usage(deep=True).sum() / 1024**2
print(f"\nTotal RAM consumida por el DataFrame: {total_ram:.2f} MB")

Index                           0.000126
stars_x                         3.659172
useful_x                        3.659172
funny_x                         3.659172
cool_x                          3.659172
text                          352.244228
date                            3.659172
review_impact_score             3.659172
review_impact_index             3.659172
review_quality                  0.457785
user_id_int                     1.829586
business_id_int                 1.829586
review_id_int                   1.829586
business_id                    32.475152
name_x                         30.853851
city                           26.781089
state                          23.327222
latitude                        3.659172
longitude                       3.659172
stars_y                         3.659172
review_count_x                  3.659172
attributes                    299.854614
categories                     53.585178
hours                         122.236801
ByAppointmentOnl

Other transformation for optimizing the dataset

In [ ]:
df_final['yelping_since'] = pd.to_datetime(df_final['yelping_since'])

In [ ]:
df_final.columns

Index(['stars_x', 'useful_x', 'funny_x', 'cool_x', 'text', 'date',
       'review_impact_score', 'review_impact_index', 'review_quality',
       'user_id_int', 'business_id_int', 'review_id_int', 'business_id',
       'name_x', 'city', 'state', 'latitude', 'longitude', 'stars_y',
       'review_count_x', 'attributes', 'categories', 'hours',
       'ByAppointmentOnly', 'BusinessAcceptsCreditCards', 'BikeParking',
       'RestaurantsPriceRange2', 'RestaurantsTakeOut', 'RestaurantsDelivery',
       'WiFi', 'WheelchairAccessible', 'HappyHour', 'OutdoorSeating', 'HasTV',
       'RestaurantsReservations', 'DogsAllowed', 'Alcohol', 'GoodForKids',
       'RestaurantsTableService', 'RestaurantsGoodForGroups', 'DriveThru',
       'NoiseLevel', 'Smoking', 'BestNight_Weekend', 'main_category', 'name_y',
       'review_count_y', 'yelping_since', 'useful_y', 'funny_y', 'cool_y',
       'elite', 'fans', 'average_stars', 'elite_count', 'raw_influence_score',
       'influence_index', 'user_type', 'fri

In [ ]:
# Master dictionary for column renaming
# This unifies user, business, and review attributes for the final dataset
column_mapping = {
    # --- Identifiers ---
    'business_id_int': 'business_id',
    'user_id_int': 'user_id',
    'review_id_int': 'review_id',
    'friends_int': 'id_friends',

    # --- Review Specific Data ---
    'stars_x': 'rating_user',       # The specific rating given by the user
    'useful_x': 'review_useful',    # Usefulness votes for the specific review
    'funny_x': 'review_funny',
    'cool_x': 'review_cool',
    'text': 'review_text',
    'date': 'review_date',
    'review_impact_index': 'index_impact_review',

    # --- Business Data ---
    'name_x': 'business_name',
    'stars_y': 'business_avg_stars',
    'review_count_x': 'business_review_count',
    'main_category': 'main_cuisine',


    # --- User Data ---
    'name_y': 'user_name',
    'average_stars': 'users_avg_stars',
    'useful_y': 'user_total_useful',
    'review_count_y': 'user_review_count',
    'funny_y': 'user_funny_reviews',
    'cool_y': 'user_cool_reviews',
    'elite': 'user_elite',
    'friends_count': 'count_friends',
    'influence_index': 'influence_user_index'
}

# Apply the unified renaming to the dataframe
df_final = df_final.rename(columns=column_mapping)

# Immediate consistency check
print("✅ Column renaming completed successfully.")
print(f"Total columns: {len(df_final.columns)}")
print("List of renamed columns:")
print(df_final.columns.tolist())

✅ Column renaming completed successfully.
Total columns: 59
List of renamed columns:
['rating_user', 'review_useful', 'review_funny', 'review_cool', 'review_text', 'review_date', 'review_impact_score', 'index_impact_review', 'review_quality', 'user_id', 'business_id', 'review_id', 'business_id', 'business_name', 'city', 'state', 'latitude', 'longitude', 'business_avg_stars', 'business_review_count', 'attributes', 'categories', 'hours', 'ByAppointmentOnly', 'BusinessAcceptsCreditCards', 'BikeParking', 'RestaurantsPriceRange2', 'RestaurantsTakeOut', 'RestaurantsDelivery', 'WiFi', 'WheelchairAccessible', 'HappyHour', 'OutdoorSeating', 'HasTV', 'RestaurantsReservations', 'DogsAllowed', 'Alcohol', 'GoodForKids', 'RestaurantsTableService', 'RestaurantsGoodForGroups', 'DriveThru', 'NoiseLevel', 'Smoking', 'BestNight_Weekend', 'main_cuisine', 'user_name', 'user_review_count', 'yelping_since', 'user_total_useful', 'user_funny_reviews', 'user_cool_reviews', 'user_elite', 'fans', 'users_avg_stars

In [ ]:
df_final_2 = df_final[['review_id',
                       'business_id',
                       'user_id',
                       'city',
                       'rating_user',
                       'review_text',
                       'review_date',
                       'review_impact_score',
                       'index_impact_review',
                       'review_quality',
                       'business_name',
                       'latitude',
                       'longitude',
                       'user_name',
                       'id_friends',
                       'main_cuisine',
                       'influence_user_index',
                       'user_type',
                       'ByAppointmentOnly',
                       'BusinessAcceptsCreditCards',
                       'BikeParking',
                       'RestaurantsPriceRange2',
                       'RestaurantsTakeOut',
                       'RestaurantsDelivery',
                        'WiFi',
                       'WheelchairAccessible',
                       'HappyHour',
                       'OutdoorSeating',
                        'HasTV',
                       'RestaurantsReservations',
                       'DogsAllowed',
                       'Alcohol',
                        'GoodForKids',
                       'RestaurantsTableService',
                       'RestaurantsGoodForGroups',
                        'DriveThru',
                       'NoiseLevel',
                       'Smoking',
                       'BestNight_Weekend']]

In [ ]:
# Si la columna es un string "[123, 456]" (común al cargar desde CSV/JSON)
import ast
def safe_len(x):
    try:
        if isinstance(x, str):
            # Convierte el string "[1, 2]" en una lista real [1, 2]
            return len(ast.literal_eval(x))
        return len(x) if isinstance(x, list) else 0
    except:
        return 0

df_final_2['count_friends'] = df_final_2['id_friends'].apply(safe_len)

# Verificamos los primeros resultados
print(df_final_2[['user_id', 'id_friends', 'count_friends']].head())

   user_id                                         id_friends  count_friends
0    22392                                            [31710]              1
1    13301  [32880, 47617, 31925, 7028, 447, 74410, 3077, ...             69
2    21483  [48194, 44204, 72308, 55404, 30226, 66696, 538...             10
3    14528  [30510, 33468, 81558, 32001, 74410, 30476, 324...             59
4    20074  [15740, 8585, 30267, 29214, 33953, 46133, 1098...             50


/tmp/ipython-input-1272701810.py:12: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
xfdgfdgjgf

NameError: name 'xfdgfdgjgf' is not defined

In [ ]:
df_final_2.info()

In [ ]:
print(df_final_2.shape)
print(df_final_2.columns)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Define la ruta dentro de tu Drive (asegúrate de que la carpeta exista)
ruta_drive = '/content/drive/My Drive/df_yelp_final_4.csv'

# Guardar como CSV
df_final_2.to_csv(ruta_drive, index=False)

print(f"✅ File saved: {ruta_drive}")

fin :)

In [ ]:
import json
import pandas as pd

# 1. Obtener solo los IDs de los negocios que son restaurantes
# Esto es muy rápido y consume poca memoria
def get_restaurant_ids(path):
    restaurant_ids = set()
    business_file = f"{path}/yelp_academic_dataset_business.json"

    print("Filtrando IDs de restaurantes...")
    with open(business_file, encoding='utf-8') as f:
        for line in f:
            biz = json.loads(line)
            # Solo guardamos el ID si el negocio tiene la categoría 'Restaurants'
            if biz['categories'] and 'Restaurants' in biz['categories']:
                restaurant_ids.add(biz['business_id'])
    return restaurant_ids

# 2. Cargar reseñas pero filtrando en el momento (Streaming)
def load_filtered_reviews(path, target_ids, n_rows_to_keep=200000):
    data = []
    count = 0
    review_file = f"{path}/yelp_academic_dataset_review.json"

    print(f"Buscando {n_rows_to_keep} reseñas de restaurantes...")
    with open(review_file, encoding='utf-8') as f:
        for line in f:
            review = json.loads(line)
            # Solo la agregamos a la lista si pertenece a un restaurante de nuestra lista
            if review['business_id'] in target_ids:
                data.append(review)
                count += 1

            # Si ya llegamos al número deseado, paramos de leer el archivo
            if count >= n_rows_to_keep:
                break

    return pd.DataFrame(data)

# --- EJECUCIÓN ---

# Paso A: Conseguir los IDs
rest_ids = get_restaurant_ids(path)
print(f"Se encontraron {len(rest_ids)} restaurantes en total.")

# Paso B: Cargar las reseñas (aquí puedes pedir 200.000 o 300.000 sin problemas)
# Como ahora TODAS son de restaurantes, no desperdiciarás RAM
df_reviews_rest = load_filtered_reviews(path, rest_ids, n_rows_to_keep=200000)

print(f"DataFrame creado con éxito. Tamaño: {df_reviews_rest.shape}")
print(df_reviews_rest.head())

In [ ]:
df_business.info()

In [ ]:
# 1. Unir Reseñas con Restaurantes
# Usamos 'inner' para asegurar que solo queden reseñas de locales que filtramos como 'Restaurants'
df_merged = pd.merge(
    df_reviews[['user_id', 'business_id', 'stars', 'text']],
    df_business[['business_id', 'name', 'categories', 'city', 'state']],
    on='business_id',
    how='inner'
)

# Renombrar 'stars' de la reseña para no confundirla con el promedio del negocio
df_merged = df_merged.rename(columns={'stars': 'user_rating'})

# 2. Cargar y unir Usuarios (Opcional pero recomendado para modelos avanzados)
df_users = load_yelp_sample('yelp_academic_dataset_user.json',)
df_final = pd.merge(
    df_merged,
    df_users[['user_id', 'review_count', 'average_stars', 'friends']],
    on='user_id',
    how='inner'
)

print(f"Dataset final listo: {df_final.shape[0]} interacciones.")

In [ ]:
df_final.info()

In [ ]:
df_final.head()

In [ ]:
# Quedarse con usuarios que tengan al menos 3 reseñas
user_counts = df_final['user_id'].value_counts()
df_final = df_final[df_final['user_id'].isin(user_counts[user_counts >= 0].index)]

# Quedarse con restaurantes que tengan al menos 5 reseñas
business_counts = df_final['business_id'].value_counts()
df_final = df_final[df_final['business_id'].isin(business_counts[business_counts >= 5].index)]

In [ ]:
#cantidad de reseñas por businessid ordenado de forma ascendente
df_final.groupby('business_id')['user_id'].count().sort_values(ascending=False)

In [ ]:
df_final.info()

-- Analisis exploratorio

In [ ]:
# Relevancia
# Formato variables
# Valores faltantes, duplicados y outliers
# Confianza
#Tamaño
# Representativo

In [ ]:
# Crear matriz: Filas (Usuarios) x Columnas (Restaurantes)
matriz_recomendacion = df_final.pivot_table(index='user_id', columns='name', values='user_rating')

In [ ]:
matriz_recomendacion

In [ ]:
df_final